In [1]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [3]:
df = pd.read_parquet('../data/processed/cleaned_data.parquet')

In [4]:
# df = pd.read_parquet('https://github.com/iamnitishsah/FoodOps.AI/raw/refs/heads/main/DeliveryOps/data/processed/cleaned_data.parquet')

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 195788 entries, 0 to 195787
Data columns (total 25 columns):
 #   Column                                        Non-Null Count   Dtype                     
---  ------                                        --------------   -----                     
 0   market_id                                     195788 non-null  float64                   
 1   created_at                                    195788 non-null  datetime64[ns]            
 2   actual_delivery_time                          195788 non-null  datetime64[ns]            
 3   store_id                                      195788 non-null  int64                     
 4   store_primary_category                        195788 non-null  object                    
 5   order_protocol                                195788 non-null  float64                   
 6   total_items                                   195788 non-null  int64                     
 7   subtotal                     

In [6]:
df.sample(10)

,market_id,created_at,actual_delivery_time,store_id,store_primary_category,order_protocol,total_items,subtotal,num_distinct_items,min_item_price,...,estimated_store_to_consumer_driving_duration,market_id_imputed,store_category_imputed,total_onshift_dashers_imputed,total_busy_dashers_imputed,total_outstanding_orders_imputed,total_delivery_duration,created_at_local,order_hour,order_day_of_week
59802,5.0,2015-02-04 19:26:38,2015-02-04 20:09:25,3045,alcohol,2.0,3,2032,3,334,...,727.0,False,False,False,False,False,2567.0,2015-02-04 11:26:38-08:00,11,2
60368,6.0,2015-02-11 04:03:17,2015-02-11 04:54:43,6090,breakfast,1.0,3,4400,3,700,...,145.0,False,False,True,True,True,3086.0,2015-02-10 20:03:17-08:00,20,1
51123,1.0,2015-01-25 03:41:25,2015-01-25 04:46:06,4555,burger,4.0,4,3536,3,549,...,317.0,False,False,False,False,False,3881.0,2015-01-24 19:41:25-08:00,19,5
67556,2.0,2015-02-14 22:10:04,2015-02-14 22:36:41,6145,mexican,3.0,2,1700,1,850,...,767.0,False,False,False,False,False,1597.0,2015-02-14 14:10:04-08:00,14,5
78420,2.0,2015-02-06 02:39:35,2015-02-06 03:17:54,3885,pizza,3.0,5,3995,1,799,...,647.0,False,False,False,False,False,2299.0,2015-02-05 18:39:35-08:00,18,3
107133,1.0,2015-01-29 19:15:41,2015-01-29 19:46:46,724,mediterranean,5.0,5,4738,3,300,...,477.0,False,False,False,False,False,1865.0,2015-01-29 11:15:41-08:00,11,3
6838,4.0,2015-02-05 01:50:36,2015-02-05 02:34:19,2427,mexican,3.0,3,2200,2,550,...,201.0,False,False,False,False,False,2623.0,2015-02-04 17:50:36-08:00,17,2
155642,3.0,2015-01-21 20:20:01,2015-01-21 20:52:58,2770,french,1.0,2,2690,2,1195,...,270.0,False,False,False,False,False,1977.0,2015-01-21 12:20:01-08:00,12,2
69165,6.0,2015-02-09 02:03:04,2015-02-09 02:35:35,5300,other,5.0,7,2825,7,325,...,242.0,False,False,True,True,True,1951.0,2015-02-08 18:03:04-08:00,18,6
34122,6.0,2015-02-07 04:45:51,2015-02-07 05:24:20,6177,mediterranean,2.0,1,899,1,899,...,572.0,False,False,True,True,True,2309.0,2015-02-06 20:45:51-08:00,20,4


# Feature Engineering

## Drop unneeded columns

In [7]:
# Drop market_id_imputed (real-world conditional fill, no longer needed as signal)
# Drop raw UTC created_at (superseded by created_at_local) and actual_delivery_time
# (target is already computed as total_delivery_duration; keeping actual_delivery_time
# around risks accidental leakage in later exploratory code)
df = df.drop(columns=["market_id_imputed", "store_category_imputed", "created_at", "actual_delivery_time"])
df.shape

(195788, 21)

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 195788 entries, 0 to 195787
Data columns (total 21 columns):
 #   Column                                        Non-Null Count   Dtype                     
---  ------                                        --------------   -----                     
 0   market_id                                     195788 non-null  float64                   
 1   store_id                                      195788 non-null  int64                     
 2   store_primary_category                        195788 non-null  object                    
 3   order_protocol                                195788 non-null  float64                   
 4   total_items                                   195788 non-null  int64                     
 5   subtotal                                      195788 non-null  int64                     
 6   num_distinct_items                            195788 non-null  int64                     
 7   min_item_price               

## Create busy_dasher_ratio and outstanding_order_ratio features

In [9]:
zero_onshift = df["total_onshift_dashers"] == 0
print(f"Rows with onshift==0: {zero_onshift.sum()} ({zero_onshift.mean():.4%})")

# What do busy/outstanding look like when onshift==0?
df.loc[zero_onshift, ["total_busy_dashers", "total_outstanding_orders", "total_delivery_duration"]].describe()

# Compare delivery duration for onshift==0 vs the rest
print("Median duration, onshift==0:", df.loc[zero_onshift, "total_delivery_duration"].median())
print("Median duration, onshift>0: ", df.loc[~zero_onshift, "total_delivery_duration"].median())

Rows with onshift==0: 3540 (1.8081%)


,total_busy_dashers,total_outstanding_orders,total_delivery_duration
count,3540.000000,3540.000000,3540.000000
mean,0.042938,0.116102,3257.166949
std,0.844086,0.704661,1179.006617
min,-4.000000,0.000000,960.000000
25%,0.000000,0.000000,2374.750000
50%,0.000000,0.000000,3067.000000
75%,0.000000,0.000000,3941.000000
max,29.000000,13.000000,7169.000000


Median duration, onshift==0: 3067.0
Median duration, onshift>0:  2648.0


In [10]:
# Raw ratio, NaN where onshift == 0
raw_busy_ratio = df["total_busy_dashers"] / df["total_onshift_dashers"].replace(0, np.nan)
raw_outstanding_ratio = df["total_outstanding_orders"] / df["total_onshift_dashers"].replace(0, np.nan)

# Empirical 99th percentile of the *valid* (non-zero-onshift) ratio distribution
busy_p99 = raw_busy_ratio.quantile(0.99)
outstanding_p99 = raw_outstanding_ratio.quantile(0.99)

busy_cap = busy_p99 * 2  # generous cap for the "impossible/undefined" onshift=0 case
outstanding_cap = outstanding_p99 * 2

print(f"busy_ratio p99={busy_p99:.3f} -> cap={busy_cap:.3f}")
print(f"outstanding_ratio p99={outstanding_p99:.3f} -> cap={outstanding_cap:.3f}")

df["busy_dasher_ratio"] = raw_busy_ratio.fillna(busy_cap)
df["outstanding_order_ratio"] = raw_outstanding_ratio.fillna(outstanding_cap)

# sanity checks
(df["total_onshift_dashers"] == 0).sum()
df[["busy_dasher_ratio", "outstanding_order_ratio"]].describe()

busy_ratio p99=1.964 -> cap=3.929
outstanding_ratio p99=2.286 -> cap=4.571


np.int64(3540)

,busy_dasher_ratio,outstanding_order_ratio
count,195788.000000,195788.000000
mean,1.001822,1.275728
std,0.552167,0.636951
min,-13.000000,-16.000000
25%,0.842857,0.960784
50%,0.962963,1.209777
75%,1.000000,1.466667
max,31.000000,47.000000


## Time-based train/test split

In [11]:
# Row-count-based split can slice unevenly across days when order volume
# varies by day-of-week. Switch to a calendar-day cutoff instead.
df["order_date"] = df["created_at_local"].dt.date

unique_dates = sorted(df["order_date"].unique())
print(f"Total unique dates: {len(unique_dates)}")
print(unique_dates)

Total unique dates: 28
[datetime.date(2015, 1, 21), datetime.date(2015, 1, 22), datetime.date(2015, 1, 23), datetime.date(2015, 1, 24), datetime.date(2015, 1, 25), datetime.date(2015, 1, 26), datetime.date(2015, 1, 27), datetime.date(2015, 1, 28), datetime.date(2015, 1, 29), datetime.date(2015, 1, 30), datetime.date(2015, 1, 31), datetime.date(2015, 2, 1), datetime.date(2015, 2, 2), datetime.date(2015, 2, 3), datetime.date(2015, 2, 4), datetime.date(2015, 2, 5), datetime.date(2015, 2, 6), datetime.date(2015, 2, 7), datetime.date(2015, 2, 8), datetime.date(2015, 2, 9), datetime.date(2015, 2, 10), datetime.date(2015, 2, 11), datetime.date(2015, 2, 12), datetime.date(2015, 2, 13), datetime.date(2015, 2, 14), datetime.date(2015, 2, 15), datetime.date(2015, 2, 16), datetime.date(2015, 2, 17)]


In [12]:
# Time-based train/test split (must happen before target encoding)
# Split on a calendar-day boundary so no individual day is split
# between train and test.

df = df.sort_values("created_at_local").reset_index(drop=True)

cutoff_date = pd.Timestamp("2015-02-11").date()

df["is_train"] = df["order_date"] < cutoff_date

print(f"Cutoff date: {cutoff_date}")
print(f"Train rows: {df['is_train'].sum()} ({df['is_train'].mean():.2%})")
print(f"Test rows:  {(~df['is_train']).sum()} ({(~df['is_train']).mean():.2%})")
print(
    f"Train date range: "
    f"{df.loc[df['is_train'], 'created_at_local'].min()} -> "
    f"{df.loc[df['is_train'], 'created_at_local'].max()}"
)
print(
    f"Test date range:  "
    f"{df.loc[~df['is_train'], 'created_at_local'].min()} -> "
    f"{df.loc[~df['is_train'], 'created_at_local'].max()}"
)

Cutoff date: 2015-02-11
Train rows: 143468 (73.28%)
Test rows:  52320 (26.72%)
Train date range: 2015-01-21 07:22:03-08:00 -> 2015-02-10 22:00:27-08:00
Test date range:  2015-02-11 06:41:00-08:00 -> 2015-02-17 22:00:44-08:00


In [13]:
# Sanity check: does the test window span a weekend, and is day-of-week representation reasonable in both splits?
train_dow = df.loc[df["is_train"], "order_day_of_week"].value_counts(normalize=True).sort_index()
test_dow = df.loc[~df["is_train"], "order_day_of_week"].value_counts(normalize=True).sort_index()

pd.DataFrame({"train": train_dow, "test": test_dow})

,train,test
order_day_of_week,,
0,0.127785,0.117469
1,0.123658,0.116648
2,0.127673,0.127733
3,0.135145,0.138093
4,0.169689,0.163742
5,0.168212,0.163838
6,0.147838,0.172477


## Target encoding: store_primary_category

In [14]:
# Target encoding: store_primary_category (fit on train only)
# Smoothed target encoding to avoid overfitting on rare/low-count categories.
# Formula: (category_mean * n + global_mean * smoothing) / (n + smoothing)
# Higher smoothing pulls sparse categories harder toward the global mean.

SMOOTHING = 20  # weight of the prior; tune later if needed

global_mean = df.loc[df["is_train"], "total_delivery_duration"].mean()

category_stats = (
    df.loc[df["is_train"]]
    .groupby("store_primary_category")["total_delivery_duration"]
    .agg(["mean", "count"])
)

category_stats["smoothed_mean"] = (
        (category_stats["mean"] * category_stats["count"] + global_mean * SMOOTHING)
        / (category_stats["count"] + SMOOTHING)
)

category_encoding_map = category_stats["smoothed_mean"].to_dict()

# Apply to full dataframe (train and test both use the train-fit map).
# Any category in test not seen in train falls back to global_mean.
df["store_category_target_enc"] = (
    df["store_primary_category"]
    .map(category_encoding_map)
    .fillna(global_mean)
)

print(f"Global mean (train): {global_mean:.1f}")
print(f"Categories encoded: {len(category_encoding_map)}")
category_stats.sort_values("count").head(10)

Global mean (train): 2804.8
Categories encoded: 75


,mean,count,smoothed_mean
store_primary_category,,,
alcohol-plus-food,3046.000000,1,2816.278595
chocolate,2048.000000,1,2768.754786
belgian,2710.000000,1,2800.278595
indonesian,2445.000000,2,2772.084114
lebanese,3027.285714,7,2862.475945
african,3064.333333,9,2885.339673
european,2691.363636,11,2764.543565
russian,3141.153846,13,2937.298500
cheese,3079.000000,15,2922.310014


In [15]:
# Verify no leakage: encoding values should be identical whether computed
# from train alone or from train+test slices of the map (they're the same map)
df[["store_primary_category", "store_category_target_enc"]].drop_duplicates().sort_values(
    "store_category_target_enc").head()
df[["store_primary_category", "store_category_target_enc"]].drop_duplicates().sort_values(
    "store_category_target_enc").tail()

,store_primary_category,store_category_target_enc
1433,gluten-free,2553.854770
9,fast,2596.082899
1227,convenience-store,2605.682915
6,mexican,2631.689728
11,sandwich,2635.102293


,store_primary_category,store_category_target_enc
702,cajun,3078.623452
1101,burmese,3083.581147
840,brazilian,3122.476751
1032,tapas,3125.372768
405,caribbean,3175.669374


## Cyclical encoding: order_hour and order_day_of_week

In [16]:
# Cyclical encoding: order_hour and order_day_of_week
# Raw integers imply hour=23 is far from hour=0, which is wrong for delivery
# timing (late-night orders near midnight should be treated as adjacent hours).
df["hour_sin"] = np.sin(2 * np.pi * df["order_hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["order_hour"] / 24)

df["dow_sin"] = np.sin(2 * np.pi * df["order_day_of_week"] / 7)
df["dow_cos"] = np.cos(2 * np.pi * df["order_day_of_week"] / 7)

df[["order_hour", "hour_sin", "hour_cos", "order_day_of_week", "dow_sin", "dow_cos"]].sample(8)

,order_hour,hour_sin,hour_cos,order_day_of_week,dow_sin,dow_cos
92338,10,0.500000,-8.660254e-01,2,0.974928,-0.222521
38276,18,-1.000000,-1.836970e-16,0,0.000000,1.000000
86498,11,0.258819,-9.659258e-01,1,0.781831,0.623490
45906,11,0.258819,-9.659258e-01,2,0.974928,-0.222521
149024,19,-0.965926,2.588190e-01,2,0.974928,-0.222521
90134,18,-1.000000,-1.836970e-16,1,0.781831,0.623490
165769,21,-0.707107,7.071068e-01,4,-0.433884,-0.900969
109625,17,-0.965926,-2.588190e-01,4,-0.433884,-0.900969


## Target transform candidates

In [17]:
# Target transform candidates (decide log vs. raw empirically in model_training.ipynb)
df["target_raw"] = df["total_delivery_duration"]
df["target_log1p"] = np.log1p(df["total_delivery_duration"])

df[["target_raw", "target_log1p"]].describe()

,target_raw,target_log1p
count,195788.000000,195788.000000
mean,2826.496302,7.885499
std,1016.054907,0.352173
min,223.000000,5.411646
25%,2101.000000,7.650645
50%,2654.000000,7.884200
75%,3366.000000,8.121777
max,7196.000000,8.881420


## Final feature set review before export

In [18]:
# Drop the now-redundant intermediate columns:
# - store_primary_category: superseded by store_category_target_enc
# - order_hour, order_day_of_week: superseded by sin/cos pairs
# - created_at_local: was only needed for the split; a GBDT gets no use from a raw timestamp, and keeping it around risks accidental leakage if someone derives new time features downstream without re-checking the split
# - total_delivery_duration: kept as target_raw / target_log1p instead

final_drop_cols = [
    "store_primary_category",
    "order_hour",
    "order_day_of_week",
    "created_at_local",
    "total_delivery_duration",
]
df = df.drop(columns=final_drop_cols)

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 195788 entries, 0 to 195787
Data columns (total 27 columns):
 #   Column                                        Non-Null Count   Dtype  
---  ------                                        --------------   -----  
 0   market_id                                     195788 non-null  float64
 1   store_id                                      195788 non-null  int64  
 2   order_protocol                                195788 non-null  float64
 3   total_items                                   195788 non-null  int64  
 4   subtotal                                      195788 non-null  int64  
 5   num_distinct_items                            195788 non-null  int64  
 6   min_item_price                                195788 non-null  int64  
 7   max_item_price                                195788 non-null  int64  
 8   total_onshift_dashers                         195788 non-null  float64
 9   total_busy_dashers                            19

In [19]:
df.sample(10)

,market_id,store_id,order_protocol,total_items,subtotal,num_distinct_items,min_item_price,max_item_price,total_onshift_dashers,total_busy_dashers,...,outstanding_order_ratio,order_date,is_train,store_category_target_enc,hour_sin,hour_cos,dow_sin,dow_cos,target_raw,target_log1p
11912,4.0,1229,2.0,2,2948,2,1399,1549,55.0,56.0,...,1.054545,2015-01-22,True,2830.528121,-8.660254e-01,5.000000e-01,0.433884,-0.900969,1667.0,7.419381
111856,1.0,6500,1.0,5,6910,4,675,1795,20.0,15.0,...,0.800000,2015-02-06,True,2631.689728,-9.659258e-01,2.588190e-01,-0.433884,-0.900969,3462.0,8.149891
5423,2.0,2272,5.0,2,2400,1,1200,1200,63.0,68.0,...,1.301587,2015-01-21,True,2631.689728,-8.660254e-01,5.000000e-01,0.974928,-0.222521,1639.0,7.402452
147624,5.0,4161,5.0,1,1075,1,1000,1000,26.0,5.0,...,0.538462,2015-02-11,False,2728.575827,-1.000000e+00,-1.836970e-16,0.974928,-0.222521,3507.0,8.162801
158673,5.0,2759,1.0,2,2445,2,1050,1395,27.0,24.0,...,1.185185,2015-02-13,False,2787.057791,1.224647e-16,-1.000000e+00,-0.433884,-0.900969,3065.0,8.028129
34038,1.0,3888,3.0,11,2991,4,199,599,9.0,8.0,...,0.777778,2015-01-25,True,2631.689728,-7.071068e-01,7.071068e-01,-0.781831,0.623490,5461.0,8.605570
24939,2.0,3732,5.0,3,1325,3,275,550,110.0,115.0,...,1.490909,2015-01-24,True,2785.783919,-1.000000e+00,-1.836970e-16,-0.974928,-0.222521,3119.0,8.045588
63554,2.0,2027,2.0,3,4497,3,499,699,45.0,40.0,...,1.977778,2015-01-30,True,2987.599722,-9.659258e-01,2.588190e-01,-0.433884,-0.900969,5340.0,8.583168
139836,1.0,6263,5.0,7,6685,5,600,1195,23.0,24.0,...,1.043478,2015-02-10,True,2987.599722,-8.660254e-01,-5.000000e-01,0.781831,0.623490,3322.0,8.108623
53907,4.0,2499,3.0,5,1350,3,200,350,109.0,94.0,...,1.394495,2015-01-29,True,3045.475141,-9.659258e-01,-2.588190e-01,0.433884,-0.900969,1805.0,7.498870


## Export processed data

In [20]:
# Export
df.to_csv("../data/processed/processed_data.csv", index=False)
df.to_parquet("../data/processed/processed_data.parquet", index=False)
print(f"Exported {df.shape[0]} rows, {df.shape[1]} columns")

Exported 195788 rows, 27 columns
